In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec


import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils


def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
town = "Bonn"
objective = "cases_and_conc"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results"

phase_cut_date = "2023-03-15"

In [ ]:
# load pred
seir_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_SEIR.npz")
R_eff_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_R_eff.npz")
#shedding_curve_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_shedding_curve.npz")
underreporting_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_underreporting_rate.npz")

In [ ]:

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": "Bonn",
            "sampling_area": "North_South",
            "project": "both", # one of ESI_CorA, AMELAG
            "max_precipitation_subsetting": None, # one of None, dry, light_rain
            "substance_normalization": "flow", # one of None, PMMoV, flow
            "gene_target": "N1", # one of N1, N2
            "log_scale": True, # this only considers WW measurements, not case counts
        },

        "E0": 862.857, 
        "I0": 1294.286,
        "R0": 162092.04, # 92% of pop, based on https://www.rki.de/DE/Themen/Infektionskrankheiten/Infektionskrankheiten-A-Z/C/COVID-19-Pandemie/AK-Studien/Ergebnisse.html
        "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "underreporting_model": "monotone_increasing"
}
hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_{objective}/hparams.json"
with open(hparams_path) as f:
        hparams = json.load(f)
base_config.update(hparams)
data = optimization_utils.two_phase_integrative_model_load_data(base_config)



In [ ]:

fig = plt.figure(figsize=(15.1, 4.6), layout="constrained")  # or constrained_layout=True on older Matplotlib
# Root 2x1 grid; increase row spacing with hspace
G = fig.add_gridspec(nrows=2, ncols=1, hspace=3.5)

# Top row: 4 equal subplots
top = G[0].subgridspec(1, 4, wspace=0.25)
ax1 = fig.add_subplot(top[0, 0])
ax2 = fig.add_subplot(top[0, 1])
ax3 = fig.add_subplot(top[0, 2])
ax4 = fig.add_subplot(top[0, 3])

# Bottom row: 3 plots with a spacer column between the 6th and 7th plots
# width_ratios reserves extra blank room between ax6 and ax7
bottom = G[1].subgridspec(1, 4, wspace=0.25, width_ratios=[1, 1, 0.15, 1])
ax5 = fig.add_subplot(bottom[0, 0])
ax6 = fig.add_subplot(bottom[0, 1])
# optional: keep the spacer cell explicitly empty
# _spacer = fig.add_subplot(bottom[0, 2]); _spacer.axis("off")
ax7 = fig.add_subplot(bottom[0, 3])


seir_quantiles = {q: jnp.quantile(seir_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

SEIR_median = seir_quantiles[0.5]


labels = [r"$S$", r"$E$", r"$I$", r"$R$"]
colors = ["goldenrod", "peru", "darkgoldenrod", "goldenrod"]

axs = [ax1, ax2, ax3, ax4]
for i in range(4):
    axs[i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
    axs[i].plot(data["dates_all"][7:], SEIR_median[7:, i], color="goldenrod", label="Median")
    SEIR_low  = seir_quantiles[0.25]
    SEIR_high = seir_quantiles[0.75]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.45, label="50% CI")
    SEIR_low  = seir_quantiles[0.05]
    SEIR_high = seir_quantiles[0.95]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.3, label="90% CI")
    SEIR_low  = seir_quantiles[0.025]
    SEIR_high = seir_quantiles[0.975]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.15, label="95% CI")

    # axs[i].set_title(labels[i])
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
    axs[i].tick_params(axis='x', rotation=45)

    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-3, 3))
    axs[i].yaxis.set_major_formatter(formatter)
    axs[i].set_ylabel(f"{labels[i]}  [#]")

# plot R_eff
R_eff_quantiles = {q: jnp.quantile(R_eff_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

Rt_median = R_eff_quantiles[0.5]

ax5.axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
ax5.plot(data["dates_all"][7:], Rt_median[7:], c="goldenrod", label="Median")

Rt_low  = R_eff_quantiles[0.25]
Rt_high = R_eff_quantiles[0.75]
ax5.fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.45, label="50% CI")

Rt_low  = R_eff_quantiles[0.05]
Rt_high = R_eff_quantiles[0.95]
ax5.fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.3, label="90% CI")

Rt_low  = R_eff_quantiles[0.025]
Rt_high = R_eff_quantiles[0.975]
ax5.fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.15, label="95% CI")


ax5.axhline(1.0, color="#A1A1A1", linestyle='--')
ax5.set_ylabel(r"$R_t$")
ax5.tick_params(axis='x', rotation=45)
ax5.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

# visualize underreporting rate

underreporting_quantiles = {q: jnp.quantile(underreporting_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

underreporting_median = underreporting_quantiles[0.5]

ax6.axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
ax6.plot(data["dates_all"][7:], (1-underreporting_median[7:])*100, c="goldenrod", label="Median")

Rt_low  = underreporting_quantiles[0.25]
Rt_high = underreporting_quantiles[0.75]
ax6.fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.45, label="50% CI")

Rt_low  = underreporting_quantiles[0.05]
Rt_high = underreporting_quantiles[0.95]
ax6.fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.3, label="90% CI")

Rt_low  = underreporting_quantiles[0.025]
Rt_high = underreporting_quantiles[0.975]
ax6.fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.15, label="95% CI")


ax6.set_ylabel("Reporting rate [%]")
ax6.tick_params(axis='x', rotation=45)
ax6.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
ax6.legend()

# add shedding kernel curve
"""
s = jnp.arange(0, base_config["T_max"]+base_config["dt"], base_config["dt"])

shedding_curve_quantiles = {q: jnp.quantile(shedding_curve_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

shedding_median = shedding_curve_quantiles[0.5]

ax7.plot(s, shedding_median, c="black", label="Median")

Rt_low  = shedding_curve_quantiles[0.25]
Rt_high = shedding_curve_quantiles[0.75]
ax7.fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.45, label="50% CI")

Rt_low  = shedding_curve_quantiles[0.05]
Rt_high = shedding_curve_quantiles[0.95]
ax7.fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.3, label="90% CI")

Rt_low  = shedding_curve_quantiles[0.025]
Rt_high = shedding_curve_quantiles[0.975]
ax7.fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.15, label="95% CI")
ax7.set_xlabel("Days since becoming infected")
ax7.set_ylabel(r"Shedding kernel")
ax7.legend()
"""

plt.savefig("Bonn_hidden_states.png", dpi=300)

In [ ]:
print(f"The underreporting reaches in the end a median value of {(1-underreporting_quantiles[0.5][-1])*100:.2f}%")

In [ ]:

fig, axs = plt.subplots(figsize=(15.1, 2.3), ncols=4)  # or constrained_layout=True on older Matplotlib


seir_quantiles = {q: jnp.quantile(seir_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
SEIR_median = seir_quantiles[0.5]


labels = [r"$S$", r"$E$", r"$I$", r"$R$"]
colors = ["goldenrod", "peru", "darkgoldenrod", "goldenrod"]


for i in range(4):
    axs[i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
    axs[i].plot(data["dates_all"][7:], SEIR_median[7:, i], color="goldenrod", label="Median")
    SEIR_low  = seir_quantiles[0.25]
    SEIR_high = seir_quantiles[0.75]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.45, label="50% CI")
    SEIR_low  = seir_quantiles[0.05]
    SEIR_high = seir_quantiles[0.95]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.3, label="90% CI")
    SEIR_low  = seir_quantiles[0.025]
    SEIR_high = seir_quantiles[0.975]
    axs[i].fill_between(data["dates_all"][7:], SEIR_low[7:, i], SEIR_high[7:, i], color="goldenrod", alpha=0.15, label="95% CI")

    # axs[i].set_title(labels[i])
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
    axs[i].tick_params(axis='x', rotation=45)

    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-3, 3))
    axs[i].yaxis.set_major_formatter(formatter)
    axs[i].set_ylabel(f"{labels[i]}  [#]")

plt.tight_layout()
plt.savefig("Bonn_hidden_states.png", dpi=300)

In [ ]:

fig, axs = plt.subplots(nrows=2, ncols=1, sharex=True, sharey=False, figsize=(4.88, 4), dpi=300, constrained_layout=True)

# plot R_eff
R_eff_quantiles = {q: jnp.quantile(R_eff_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

Rt_median = R_eff_quantiles[0.5]

axs[0].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
axs[0].plot(data["dates_all"][7:], Rt_median[7:], c="goldenrod", label="Median")

Rt_low  = R_eff_quantiles[0.25]
Rt_high = R_eff_quantiles[0.75]
axs[0].fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.45, label="50% CI")

Rt_low  = R_eff_quantiles[0.05]
Rt_high = R_eff_quantiles[0.95]
axs[0].fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.3, label="90% CI")

Rt_low  = R_eff_quantiles[0.025]
Rt_high = R_eff_quantiles[0.975]
axs[0].fill_between(data["dates_all"][7:], Rt_low[7:], Rt_high[7:], color="goldenrod", alpha=0.15, label="95% CI")


axs[0].axhline(1.0, color="#A1A1A1", linestyle='--')
axs[0].set_ylabel(r"$R_t$")
axs[0].tick_params(axis='x', rotation=45)
axs[0].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
axs[0].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

# visualize underreporting rate

underreporting_quantiles = {q: jnp.quantile(underreporting_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

underreporting_median = underreporting_quantiles[0.5]

axs[1].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")
axs[1].plot(data["dates_all"][7:], (1-underreporting_median[7:])*100, c="goldenrod", label="Median")

Rt_low  = underreporting_quantiles[0.25]
Rt_high = underreporting_quantiles[0.75]
axs[1].fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.45, label="50% CI")

Rt_low  = underreporting_quantiles[0.05]
Rt_high = underreporting_quantiles[0.95]
axs[1].fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.3, label="90% CI")

Rt_low  = underreporting_quantiles[0.025]
Rt_high = underreporting_quantiles[0.975]
axs[1].fill_between(data["dates_all"][7:], (1-Rt_low[7:])*100, (1-Rt_high[7:])*100, color="goldenrod", alpha=0.15, label="95% CI")


axs[1].set_ylabel("Reporting rate [%]")
axs[1].tick_params(axis='x', rotation=45)
axs[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
axs[1].legend()

plt.savefig("Figure2_e.png", dpi=300)
